# Kalshi In-Play Brier Score (T-60 min)

Uses the Kalshi market price **1 hour before match end** as the predicted probability.
This approximates the odds available after set 1 has finished — the point at which
we'd realistically be placing in-play bets.

- Average match duration: ~115 min → T-60 lands at roughly set 1 / early set 2
- Comparison baseline: opening price Brier = 0.2293

In [1]:
import glob
import os
from collections import defaultdict
from datetime import timedelta

import numpy as np
import pandas as pd

DATA_DIR = 'kalshi_match_data_2026'

In [2]:
# ── Group files by match ID, pick one per match ───────────────────────────────
all_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_candles.csv')))

by_match = defaultdict(list)
for f in all_files:
    base = os.path.basename(f)
    match_id = base.rsplit('-', 1)[0]
    by_match[match_id].append(f)

chosen = {mid: sorted(files)[0] for mid, files in by_match.items()}

print(f'Total files   : {len(all_files)}')
print(f'Unique matches: {len(chosen)}')

Total files   : 1446
Unique matches: 886


In [3]:
def parse_candle_file(path):
    """
    Returns (t60_prob, outcome) or None.
    t60_prob : 'previous' from the row closest to 60 min before match end
    outcome  : last non-null close (or previous), rounded to 0/1
    """
    df = pd.read_csv(path)
    if df.empty:
        return None

    df['ts'] = pd.to_datetime(df['end_period_ts'])
    df = df.sort_values('ts').reset_index(drop=True)

    last_ts = df['ts'].iloc[-1]
    first_ts = df['ts'].iloc[0]

    # Need at least 61 minutes of data to get a valid T-60 reading
    if (last_ts - first_ts).total_seconds() < 61 * 60:
        return None

    # Outcome: last non-null close, fallback to last previous
    last_close = df['close'].dropna()
    if not last_close.empty:
        last_val = last_close.iloc[-1]
    else:
        last_val = df['previous'].dropna().iloc[-1]
    outcome = 1 if last_val >= 0.5 else 0

    # T-60: last row with ts <= last_ts - 60 min
    target_ts = last_ts - timedelta(minutes=60)
    candidates = df[df['ts'] <= target_ts]
    if candidates.empty:
        return None

    t60_row = candidates.iloc[-1]
    t60_prob = t60_row['previous']
    if pd.isna(t60_prob):
        return None

    minutes_before_end = (last_ts - t60_row['ts']).total_seconds() / 60
    return float(t60_prob), outcome, float(minutes_before_end)


records = []
skipped = 0

for match_id, path in chosen.items():
    result = parse_candle_file(path)
    if result is None:
        skipped += 1
        continue
    t60_prob, outcome, mins_before = result
    records.append({
        'match_id':    match_id,
        'file':        os.path.basename(path),
        't60_prob':    t60_prob,
        'outcome':     outcome,
        'mins_before': mins_before,
        'brier':       (t60_prob - outcome) ** 2,
    })

df = pd.DataFrame(records)
print(f'Parsed: {len(df)}  |  Skipped (< 61 min data): {skipped}')
print(f'Avg minutes before end at T-60 row: {df["mins_before"].mean():.1f}')
print(df[['file','t60_prob','outcome','brier']].head(8).to_string(index=False))

Parsed: 883  |  Skipped (< 61 min data): 3
Avg minutes before end at T-60 row: 60.1
                                              file  t60_prob  outcome  brier
KXATPCHALLENGERMATCH-26FEB01BOUSCH-BOU_candles.csv      0.20        0 0.0400
KXATPCHALLENGERMATCH-26FEB01MOCHAZ-HAZ_candles.csv      0.65        1 0.1225
KXATPCHALLENGERMATCH-26FEB01SHITOK-SHI_candles.csv      0.68        1 0.1024
KXATPCHALLENGERMATCH-26FEB02ELLSHI-ELL_candles.csv      0.78        1 0.0484
KXATPCHALLENGERMATCH-26FEB03MORVAS-VAS_candles.csv      0.27        0 0.0729
KXATPCHALLENGERMATCH-26FEB03PRIBON-BON_candles.csv      0.42        0 0.1764
KXATPCHALLENGERMATCH-26FEB03SANCOP-SAN_candles.csv      0.19        1 0.6561
KXATPCHALLENGERMATCH-26FEB03SHAJON-JON_candles.csv      0.44        1 0.3136


In [4]:
# ── Brier score ───────────────────────────────────────────────────────────────
brier = df['brier'].mean()
print(f'Kalshi T-60 Brier score : {brier:.4f}')
print(f'Random baseline         : 0.2500')
print(f'Skill score             : {1 - brier/0.25:.4f}  (0=random, 1=perfect)')
print()
print(f'Matches analysed : {len(df)}')
print(f'Outcome=1 (won)  : {df["outcome"].sum()}  ({df["outcome"].mean()*100:.1f}%)')
print(f't60_prob mean    : {df["t60_prob"].mean():.3f}')
print(f't60_prob range   : {df["t60_prob"].min():.3f} – {df["t60_prob"].max():.3f}')

Kalshi T-60 Brier score : 0.2109
Random baseline         : 0.2500
Skill score             : 0.1566  (0=random, 1=perfect)

Matches analysed : 883
Outcome=1 (won)  : 460  (52.1%)
t60_prob mean    : 0.504
t60_prob range   : 0.010 – 0.990


In [5]:
# ── ATP vs Challenger breakdown ───────────────────────────────────────────────
df['is_challenger'] = df['match_id'].str.startswith('KXATPCHALLENGERMATCH')

for label, subset in [('ATP', df[~df['is_challenger']]), ('Challenger', df[df['is_challenger']])]:
    b = subset['brier'].mean()
    print(f'{label:12s}: {len(subset):>3} matches | Brier {b:.4f} | skill {1-b/0.25:.4f}')

ATP         : 605 matches | Brier 0.1996 | skill 0.2017
Challenger  : 278 matches | Brier 0.2354 | skill 0.0585


In [6]:
# ── Calibration: predicted prob vs actual win rate ────────────────────────────
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
df['bucket'] = pd.cut(df['t60_prob'], bins=bins)
cal = df.groupby('bucket', observed=False).agg(
    n=('outcome', 'count'),
    pred_mean=('t60_prob', 'mean'),
    actual_win_rate=('outcome', 'mean')
).round(3)
print('Calibration (T-60 prob vs actual win rate):')
print(cal[cal['n'] > 0].to_string())

Calibration (T-60 prob vs actual win rate):
              n  pred_mean  actual_win_rate
bucket                                     
(0.0, 0.1]   36      0.065            0.111
(0.1, 0.2]   79      0.157            0.203
(0.2, 0.3]   98      0.259            0.367
(0.3, 0.4]  109      0.354            0.431
(0.4, 0.5]  125      0.452            0.456
(0.5, 0.6]  123      0.562            0.569
(0.6, 0.7]  105      0.659            0.619
(0.7, 0.8]   91      0.750            0.714
(0.8, 0.9]   74      0.848            0.811
(0.9, 1.0]   43      0.941            0.930


In [7]:
# ── Probability distribution: compare opening vs T-60 ────────────────────────
bins2 = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
counts, _ = np.histogram(df['t60_prob'], bins=bins2)
print('Distribution of T-60 probabilities (expect more extreme than opening):')
for i, c in enumerate(counts):
    lo, hi = bins2[i], bins2[i+1]
    bar = '|' * c
    print(f'  {lo:.1f}–{hi:.1f}  {c:>4}  {bar}')

Distribution of T-60 probabilities (expect more extreme than opening):
  0.0–0.1    31  |||||||||||||||||||||||||||||||
  0.1–0.2    74  ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  0.2–0.3    94  ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  0.3–0.4   107  |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  0.4–0.5   130  ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  0.5–0.6   118  ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  0.6–0.7   109  |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  0.7–0.8    92  ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  0.8–0.9    76  |||||||||||||

In [8]:
# ── MEDWAW sanity check ───────────────────────────────────────────────────────
medwaw = df[df['match_id'].str.contains('MEDWAW', na=False)]
if not medwaw.empty:
    print('MEDWAW sanity check (opening prob was 0.63, Medvedev lost):')
    print(medwaw[['match_id', 't60_prob', 'outcome', 'mins_before']].to_string(index=False))
else:
    print('MEDWAW not found in this dataset')

MEDWAW sanity check (opening prob was 0.63, Medvedev lost):
                match_id  t60_prob  outcome  mins_before
KXATPMATCH-26FEB01MEDWAW      0.61        0         60.0


In [9]:
# ── Full comparison table ─────────────────────────────────────────────────────
def skill(b): return 1 - b / 0.25

print('─' * 62)
print(f'{"Model":<34}  {"Brier":>7}  {"Skill":>7}')
print('─' * 62)
print(f'{"Random baseline":<34}  {0.2500:.4f}  {skill(0.2500):>7.4f}')
print(f'{"Kalshi opening price":<34}  {0.2293:.4f}  {skill(0.2293):>7.4f}')
print(f'{"MC + YTD pre-match (clean)":<34}  {0.2216:.4f}  {skill(0.2216):>7.4f}')
print(f'{"MC post-set-1 (no BP)":<34}  {0.1591:.4f}  {skill(0.1591):>7.4f}')
print(f'{"Kalshi T-60 (this)":<34}  {brier:.4f}  {skill(brier):>7.4f}')
print('─' * 62)

──────────────────────────────────────────────────────────────
Model                                 Brier    Skill
──────────────────────────────────────────────────────────────
Random baseline                     0.2500   0.0000
Kalshi opening price                0.2293   0.0828
MC + YTD pre-match (clean)          0.2216   0.1136
MC post-set-1 (no BP)               0.1591   0.3636
Kalshi T-60 (this)                  0.2109   0.1566
──────────────────────────────────────────────────────────────


In [10]:
# ── Save results ──────────────────────────────────────────────────────────────
df.drop(columns=['bucket']).to_csv('kalshi_inplay_brier_results.csv', index=False)
print('Saved to kalshi_inplay_brier_results.csv')

Saved to kalshi_inplay_brier_results.csv
